# BikeStores DWH — 03 Validation & Testing

Notebook ini digunakan **setelah ETL SQLite selesai**.

Tujuan utamanya bukan sekadar memastikan kode berhasil dijalankan, tetapi membuktikan bahwa hasil DWH:

```text
✓ strukturnya benar
✓ grain-nya benar
✓ foreign key valid
✓ agregasi penjualan benar
✓ agregasi pengiriman benar
✓ hasil DWH dapat direkonsiliasi ke source
✓ dapat digunakan untuk query analitik
✓ stabil ketika pipeline dijalankan ulang
```

Alur:

```text
SQLite DWH
   │
   ├── Structural Validation
   ├── Row Count Validation
   ├── Referential Integrity
   ├── Grain Validation
   ├── Sales Reconciliation
   ├── Shipping Reconciliation
   ├── Business Query Sanity Test
   └── Idempotency Snapshot
           │
           ▼
      OVERALL STATUS
```


## 00 — Setup

Notebook menggunakan project Google Drive yang sama:

```text
MyDrive/BikeStores_DWH/
```

Input:

```text
database/bikestores_dwh.db
data/staging/clean/
```


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sqlite3
import json
import pandas as pd
import numpy as np
import os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/BikeStores_DWH')
else:
    PROJECT_DIR = Path.cwd() / 'BikeStores_DWH'

DB_PATH = PROJECT_DIR / 'database' / 'bikestores_dwh.db'
STAGING_CLEAN_DIR = PROJECT_DIR / 'data' / 'staging' / 'clean'
EVIDENCE_DIR = PROJECT_DIR / 'data' / 'evidence' / 'validation'
LOG_DIR = PROJECT_DIR / 'logs'

EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(
        f'Database tidak ditemukan: {DB_PATH}. '
        'Jalankan notebook 02 ETL SQLite terlebih dahulu.'
    )

print('PASS: database ditemukan')
print('DB_PATH:', DB_PATH)


## 01 — Buka Database dan Load Source Pembanding

Source pembanding menggunakan **staging clean**, bukan raw source, karena staging clean adalah input resmi ETL.


In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

required_source_files = [
    'orders.csv',
    'order_items.csv',
    'products.csv',
    'stores.csv',
]

missing = [
    f for f in required_source_files
    if not (STAGING_CLEAN_DIR / f).exists()
]

if missing:
    raise FileNotFoundError(
        'Staging clean tidak lengkap: ' + ', '.join(missing)
    )

orders = pd.read_csv(
    STAGING_CLEAN_DIR / 'orders.csv',
    parse_dates=['order_date', 'required_date', 'shipped_date']
)
order_items = pd.read_csv(
    STAGING_CLEAN_DIR / 'order_items.csv'
)
products = pd.read_csv(
    STAGING_CLEAN_DIR / 'products.csv'
)
stores = pd.read_csv(
    STAGING_CLEAN_DIR / 'stores.csv'
)

print('PASS: source pembanding berhasil dimuat')


# 02 — Test Registry

Semua pengujian akan dicatat dalam satu registry agar hasil akhirnya mudah dibaca dan dapat diekspor sebagai evidence.


In [ ]:
test_results = []

def record_test(category, test_name, passed, expected=None, actual=None, detail=''):
    test_results.append({
        'category': category,
        'test_name': test_name,
        'status': 'PASS' if bool(passed) else 'FAIL',
        'expected': expected,
        'actual': actual,
        'detail': detail
    })


# 03 — Structural Validation

Tabel yang wajib tersedia:

```text
dim_waktu
dim_toko
dim_produk
fakta_penjualan_bulanan
fakta_pengiriman_bulanan
```


In [ ]:
expected_tables = {
    'dim_waktu',
    'dim_toko',
    'dim_produk',
    'fakta_penjualan_bulanan',
    'fakta_pengiriman_bulanan',
}

table_df = pd.read_sql_query(
    '''
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    ''',
    conn
)

actual_tables = set(table_df['name'])

record_test(
    'STRUCTURE',
    'Seluruh tabel DWH tersedia',
    expected_tables.issubset(actual_tables),
    expected=sorted(expected_tables),
    actual=sorted(actual_tables)
)

table_df


## 03.1 — Cek Constraint Schema

Kita periksa bahwa tabel fakta benar-benar mempunyai foreign key.


In [ ]:
fk_sales = conn.execute(
    'PRAGMA foreign_key_list(fakta_penjualan_bulanan);'
).fetchall()

fk_shipping = conn.execute(
    'PRAGMA foreign_key_list(fakta_pengiriman_bulanan);'
).fetchall()

record_test(
    'STRUCTURE',
    'FK fakta_penjualan tersedia',
    len(fk_sales) == 3,
    expected=3,
    actual=len(fk_sales)
)

record_test(
    'STRUCTURE',
    'FK fakta_pengiriman tersedia',
    len(fk_shipping) == 2,
    expected=2,
    actual=len(fk_shipping)
)

print('FK fakta_penjualan:', len(fk_sales))
print('FK fakta_pengiriman:', len(fk_shipping))


# 04 — Row Count Validation

Dimensi yang berasal langsung dari master source harus dapat direkonsiliasi.


In [ ]:
tables = [
    'dim_waktu',
    'dim_toko',
    'dim_produk',
    'fakta_penjualan_bulanan',
    'fakta_pengiriman_bulanan',
]

row_counts = {}

for table in tables:
    row_counts[table] = conn.execute(
        f'SELECT COUNT(*) FROM {table};'
    ).fetchone()[0]

row_count_df = pd.DataFrame(
    [{'table': k, 'rows': v} for k, v in row_counts.items()]
)

record_test(
    'ROW_COUNT',
    'dim_toko = source stores',
    row_counts['dim_toko'] == len(stores),
    expected=len(stores),
    actual=row_counts['dim_toko']
)

record_test(
    'ROW_COUNT',
    'dim_produk = source products',
    row_counts['dim_produk'] == len(products),
    expected=len(products),
    actual=row_counts['dim_produk']
)

row_count_df


# 05 — Referential Integrity

SQLite menyediakan pemeriksaan langsung:

```sql
PRAGMA foreign_key_check;
```

Expected:

```text
0 violation
```


In [ ]:
fk_errors = conn.execute(
    'PRAGMA foreign_key_check;'
).fetchall()

record_test(
    'INTEGRITY',
    'Foreign key integrity',
    len(fk_errors) == 0,
    expected=0,
    actual=len(fk_errors)
)

print('Foreign key violations:', len(fk_errors))


# 06 — Grain Validation

### Fakta Penjualan

Grain:

```text
id_waktu + id_toko + id_produk
```

### Fakta Pengiriman

Grain:

```text
id_waktu + id_toko
```


In [ ]:
sales_duplicate_grain = pd.read_sql_query(
    '''
    SELECT
        id_waktu,
        id_toko,
        id_produk,
        COUNT(*) AS jumlah
    FROM fakta_penjualan_bulanan
    GROUP BY
        id_waktu,
        id_toko,
        id_produk
    HAVING COUNT(*) > 1;
    ''',
    conn
)

shipping_duplicate_grain = pd.read_sql_query(
    '''
    SELECT
        id_waktu,
        id_toko,
        COUNT(*) AS jumlah
    FROM fakta_pengiriman_bulanan
    GROUP BY
        id_waktu,
        id_toko
    HAVING COUNT(*) > 1;
    ''',
    conn
)

record_test(
    'GRAIN',
    'Grain fakta_penjualan unik',
    sales_duplicate_grain.empty,
    expected=0,
    actual=len(sales_duplicate_grain)
)

record_test(
    'GRAIN',
    'Grain fakta_pengiriman unik',
    shipping_duplicate_grain.empty,
    expected=0,
    actual=len(shipping_duplicate_grain)
)

print('Duplicate grain penjualan :', len(sales_duplicate_grain))
print('Duplicate grain pengiriman:', len(shipping_duplicate_grain))


# 07 — Sales Reconciliation

Kita hitung penjualan langsung dari source staging clean dan membandingkannya dengan DWH.

Source formula:

```text
gross
= quantity × list_price

discount
= quantity × list_price × discount

net
= quantity × list_price × (1 - discount)
```

Hanya:

```text
order_status = 4
```

yang dihitung sebagai penjualan.


In [ ]:
completed_orders = orders.loc[
    orders['order_status'] == 4,
    ['order_id']
]

source_sales = completed_orders.merge(
    order_items,
    on='order_id',
    how='inner',
    validate='one_to_many'
)

source_sales['gross'] = (
    source_sales['quantity']
    * source_sales['list_price']
)

source_sales['discount_value'] = (
    source_sales['gross']
    * source_sales['discount']
)

source_sales['net'] = (
    source_sales['gross']
    - source_sales['discount_value']
)

source_gross = float(source_sales['gross'].sum())
source_discount = float(source_sales['discount_value'].sum())
source_net = float(source_sales['net'].sum())

dwh_sales = pd.read_sql_query(
    '''
    SELECT
        SUM(total_penjualan_kotor) AS gross,
        SUM(total_diskon) AS discount_value,
        SUM(total_penjualan_bersih) AS net
    FROM fakta_penjualan_bulanan;
    ''',
    conn
).iloc[0]

dwh_gross = float(dwh_sales['gross'])
dwh_discount = float(dwh_sales['discount_value'])
dwh_net = float(dwh_sales['net'])

record_test(
    'SALES_RECONCILIATION',
    'Gross sales source = DWH',
    np.isclose(source_gross, dwh_gross, atol=0.01),
    expected=round(source_gross, 2),
    actual=round(dwh_gross, 2)
)

record_test(
    'SALES_RECONCILIATION',
    'Discount source = DWH',
    np.isclose(source_discount, dwh_discount, atol=0.01),
    expected=round(source_discount, 2),
    actual=round(dwh_discount, 2)
)

record_test(
    'SALES_RECONCILIATION',
    'Net sales source = DWH',
    np.isclose(source_net, dwh_net, atol=0.01),
    expected=round(source_net, 2),
    actual=round(dwh_net, 2)
)

record_test(
    'SALES_RECONCILIATION',
    'Gross - discount = net',
    np.isclose(
        dwh_gross - dwh_discount,
        dwh_net,
        atol=0.01
    ),
    expected=round(dwh_gross - dwh_discount, 2),
    actual=round(dwh_net, 2)
)

pd.DataFrame([
    {
        'metric': 'gross',
        'source': source_gross,
        'dwh': dwh_gross,
        'difference': source_gross - dwh_gross
    },
    {
        'metric': 'discount',
        'source': source_discount,
        'dwh': dwh_discount,
        'difference': source_discount - dwh_discount
    },
    {
        'metric': 'net',
        'source': source_net,
        'dwh': dwh_net,
        'difference': source_net - dwh_net
    },
])


# 08 — Shipping Reconciliation

Kita rekonsiliasi:

```text
total order
status order
late shipment
```


In [ ]:
source_total_orders = len(orders)

dwh_total_orders = conn.execute(
    '''
    SELECT SUM(jumlah_order)
    FROM fakta_pengiriman_bulanan;
    '''
).fetchone()[0]

record_test(
    'SHIPPING_RECONCILIATION',
    'Total order source = DWH',
    int(dwh_total_orders) == int(source_total_orders),
    expected=int(source_total_orders),
    actual=int(dwh_total_orders)
)

source_late = int(
    (
        orders['shipped_date'].notna()
        & (
            orders['shipped_date']
            > orders['required_date']
        )
    ).sum()
)

dwh_late = conn.execute(
    '''
    SELECT SUM(jumlah_order_terlambat)
    FROM fakta_pengiriman_bulanan;
    '''
).fetchone()[0]

record_test(
    'SHIPPING_RECONCILIATION',
    'Late shipment source = DWH',
    int(dwh_late) == int(source_late),
    expected=int(source_late),
    actual=int(dwh_late)
)

status_error_count = conn.execute(
    '''
    SELECT COUNT(*)
    FROM fakta_pengiriman_bulanan
    WHERE jumlah_order !=
          jumlah_order_pending
        + jumlah_order_diproses
        + jumlah_order_ditolak
        + jumlah_order_selesai;
    '''
).fetchone()[0]

record_test(
    'SHIPPING_RECONCILIATION',
    'Jumlah order = sum status',
    status_error_count == 0,
    expected=0,
    actual=int(status_error_count)
)

print('Source total order:', source_total_orders)
print('DWH total order   :', dwh_total_orders)
print('Source late       :', source_late)
print('DWH late          :', dwh_late)


# 09 — Status Distribution Reconciliation

Distribusi seluruh status di source dibandingkan dengan total measure status di DWH.


In [ ]:
source_status = (
    orders['order_status']
    .value_counts()
    .sort_index()
    .to_dict()
)

dwh_status = pd.read_sql_query(
    '''
    SELECT
        SUM(jumlah_order_pending) AS pending,
        SUM(jumlah_order_diproses) AS processing,
        SUM(jumlah_order_ditolak) AS rejected,
        SUM(jumlah_order_selesai) AS completed
    FROM fakta_pengiriman_bulanan;
    ''',
    conn
).iloc[0]

status_map = {
    1: ('Pending', 'pending'),
    2: ('Processing', 'processing'),
    3: ('Rejected', 'rejected'),
    4: ('Completed', 'completed'),
}

status_comparison = []

for status_id, (label, dwh_col) in status_map.items():
    src = int(source_status.get(status_id, 0))
    dwh = int(dwh_status[dwh_col])

    record_test(
        'STATUS_RECONCILIATION',
        f'{label} source = DWH',
        src == dwh,
        expected=src,
        actual=dwh
    )

    status_comparison.append({
        'status_id': status_id,
        'status': label,
        'source': src,
        'dwh': dwh
    })

pd.DataFrame(status_comparison)


# 10 — Query Sanity Test

DWH harus bisa menjawab kebutuhan analitik yang sudah ditentukan sejak tahap desain.

Query di bawah tidak menentukan apakah bisnis "baik" atau "buruk". Tujuannya memastikan struktur DWH dapat digunakan secara logis.


In [ ]:
sales_by_store = pd.read_sql_query(
    '''
    SELECT
        t.nama_toko,
        ROUND(
            SUM(f.total_penjualan_bersih),
            2
        ) AS total_penjualan_bersih
    FROM fakta_penjualan_bulanan AS f
    JOIN dim_toko AS t
        ON f.id_toko = t.id_toko
    GROUP BY
        t.id_toko,
        t.nama_toko
    ORDER BY
        total_penjualan_bersih DESC;
    ''',
    conn
)

record_test(
    'ANALYTIC_QUERY',
    'Query penjualan per toko menghasilkan data',
    len(sales_by_store) > 0,
    expected='> 0 rows',
    actual=len(sales_by_store)
)

sales_by_store


In [ ]:
sales_by_category = pd.read_sql_query(
    '''
    SELECT
        p.kategori,
        ROUND(
            SUM(f.total_penjualan_bersih),
            2
        ) AS total_penjualan_bersih
    FROM fakta_penjualan_bulanan AS f
    JOIN dim_produk AS p
        ON f.id_produk = p.id_produk
    GROUP BY
        p.kategori
    ORDER BY
        total_penjualan_bersih DESC;
    ''',
    conn
)

record_test(
    'ANALYTIC_QUERY',
    'Query penjualan per kategori menghasilkan data',
    len(sales_by_category) > 0,
    expected='> 0 rows',
    actual=len(sales_by_category)
)

sales_by_category


In [ ]:
shipping_by_store = pd.read_sql_query(
    '''
    SELECT
        t.nama_toko,
        SUM(f.jumlah_order) AS jumlah_order,
        SUM(f.jumlah_order_terlambat)
            AS jumlah_order_terlambat,
        ROUND(
            AVG(f.rata_rata_hari_proses),
            2
        ) AS rata_rata_hari_proses_bulanan
    FROM fakta_pengiriman_bulanan AS f
    JOIN dim_toko AS t
        ON f.id_toko = t.id_toko
    GROUP BY
        t.id_toko,
        t.nama_toko
    ORDER BY
        jumlah_order DESC;
    ''',
    conn
)

record_test(
    'ANALYTIC_QUERY',
    'Query pengiriman per toko menghasilkan data',
    len(shipping_by_store) > 0,
    expected='> 0 rows',
    actual=len(shipping_by_store)
)

shipping_by_store


# 11 — Idempotency Snapshot

Karena notebook ETL menggunakan full reload, hasil setelah ETL dijalankan ulang harus tetap sama.

Notebook validation menyimpan **fingerprint ringkas** dari DWH:

```text
row count setiap tabel
total gross
total discount
total net
total order
total late shipment
```

### Cara melakukan test idempotency

**Run pertama notebook validation:**

```text
baseline belum ada
→ baseline dibuat
```

Kemudian:

```text
1. jalankan ulang notebook 02 ETL SQLite dengan Run all
2. jalankan ulang notebook 03 Validation & Testing dengan Run all
```

Pada run kedua notebook validation:

```text
current fingerprint
        │
        ▼ compare
previous baseline
        │
        ▼
PASS / FAIL
```


In [ ]:
fingerprint = {
    'dim_waktu_rows': int(row_counts['dim_waktu']),
    'dim_toko_rows': int(row_counts['dim_toko']),
    'dim_produk_rows': int(row_counts['dim_produk']),
    'fact_sales_rows': int(row_counts['fakta_penjualan_bulanan']),
    'fact_shipping_rows': int(row_counts['fakta_pengiriman_bulanan']),
    'gross': round(float(dwh_gross), 4),
    'discount': round(float(dwh_discount), 4),
    'net': round(float(dwh_net), 4),
    'total_orders': int(dwh_total_orders),
    'late_shipments': int(dwh_late),
}

baseline_path = EVIDENCE_DIR / 'idempotency_baseline.json'

if baseline_path.exists():
    previous = json.loads(
        baseline_path.read_text(encoding='utf-8')
    )

    same = previous == fingerprint

    record_test(
        'IDEMPOTENCY',
        'Fingerprint sama dengan run sebelumnya',
        same,
        expected=previous,
        actual=fingerprint,
        detail=(
            'ETL rerun stabil'
            if same
            else 'Hasil berubah setelah rerun'
        )
    )

    print(
        'PASS: idempotency fingerprint sama.'
        if same
        else 'FAIL: fingerprint berubah.'
    )
else:
    baseline_path.write_text(
        json.dumps(fingerprint, indent=2),
        encoding='utf-8'
    )

    print('Baseline idempotency dibuat.')
    print(
        'Untuk menguji idempotency: '
        'Run all notebook 02, lalu Run all notebook 03 lagi.'
    )


# 12 — Final Test Summary

Semua hasil test dikumpulkan dalam satu tabel.

Catatan:
- pada run pertama, idempotency belum mendapat status karena baseline baru dibuat;
- setelah ETL dijalankan ulang dan notebook ini dijalankan lagi, test idempotency akan ikut muncul.


In [ ]:
results_df = pd.DataFrame(test_results)

display(results_df)

fail_count = int((results_df['status'] == 'FAIL').sum())
pass_count = int((results_df['status'] == 'PASS').sum())

print()
print('PASS:', pass_count)
print('FAIL:', fail_count)

if fail_count == 0:
    print('OVERALL STATUS: PASS')
else:
    print('OVERALL STATUS: FAIL')


# 13 — Export Evidence

Evidence utama:

```text
data/evidence/validation/
├── validation_results.csv
├── row_counts.csv
├── sales_reconciliation.csv
├── status_reconciliation.csv
├── sales_by_store.csv
├── sales_by_category.csv
├── shipping_by_store.csv
└── idempotency_baseline.json
```


In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

results_df.to_csv(
    EVIDENCE_DIR / 'validation_results.csv',
    index=False
)

row_count_df.to_csv(
    EVIDENCE_DIR / 'row_counts.csv',
    index=False
)

pd.DataFrame([
    {
        'metric': 'gross',
        'source': source_gross,
        'dwh': dwh_gross,
        'difference': source_gross - dwh_gross
    },
    {
        'metric': 'discount',
        'source': source_discount,
        'dwh': dwh_discount,
        'difference': source_discount - dwh_discount
    },
    {
        'metric': 'net',
        'source': source_net,
        'dwh': dwh_net,
        'difference': source_net - dwh_net
    },
]).to_csv(
    EVIDENCE_DIR / 'sales_reconciliation.csv',
    index=False
)

pd.DataFrame(status_comparison).to_csv(
    EVIDENCE_DIR / 'status_reconciliation.csv',
    index=False
)

sales_by_store.to_csv(
    EVIDENCE_DIR / 'sales_by_store.csv',
    index=False
)

sales_by_category.to_csv(
    EVIDENCE_DIR / 'sales_by_category.csv',
    index=False
)

shipping_by_store.to_csv(
    EVIDENCE_DIR / 'shipping_by_store.csv',
    index=False
)

validation_log = pd.DataFrame([{
    'run_id': run_id,
    'run_time_utc': datetime.now(timezone.utc).isoformat(),
    'pass_count': pass_count,
    'fail_count': fail_count,
    'overall_status': 'PASS' if fail_count == 0 else 'FAIL'
}])

validation_log.to_csv(
    LOG_DIR / f'validation_run_{run_id}.csv',
    index=False
)

print('Evidence:', EVIDENCE_DIR)
print('Log     :', LOG_DIR)


# 14 — Kriteria Tahap 09 Selesai

Tahap Validation & Testing dianggap selesai apabila:

```text
✓ structural test PASS
✓ row count test PASS
✓ foreign key test PASS
✓ grain test PASS
✓ sales reconciliation PASS
✓ shipping reconciliation PASS
✓ status reconciliation PASS
✓ analytic query test PASS
✓ idempotency test PASS
```

Karena idempotency membutuhkan perbandingan antar-run:

```text
Run notebook 03 pertama kali
        ↓
baseline dibuat

Run notebook 02 lagi
        ↓
DWH dibangun ulang

Run notebook 03 lagi
        ↓
idempotency dibandingkan
```

Jika run kedua notebook 03 menunjukkan:

```text
OVERALL STATUS: PASS
```

dan terdapat test:

```text
IDEMPOTENCY → PASS
```

maka tahap 09 selesai dan proyek dapat lanjut ke:

```text
10 — Scheduler & Automation
```


In [ ]:
conn.close()
print('Koneksi SQLite ditutup.')
